# 03c — HTTP, validation, and FastAPI — Worked answers

Compare reasoning as well as output. Close this file and repeat with changed inputs after 48 hours.

**Prerequisites:** earlier lessons in the course order. **Environment:** Install requirements/core.txt.

[Stage guide](../stages/03.md) · [Course map](../PLAN.md) · [Project](../../projects/stage03/README.md)

**Notebook time:** 2–4 hours for an initial pass; project, reading, retrieval practice, and independent transfer use the rest of the stage budget.

Learning objectives:

- Model request and response contracts
- Test HTTP errors without a live server
- Separate authentication and authorization


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets" / "financials.csv").exists()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook from inside the learning repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

def expect_error(error_type, operation):
    try:
        operation()
    except error_type:
        return
    raise AssertionError(f"Expected {error_type.__name__}")

print("Course root:", ROOT)


## Before the exercises: Following an HTTP request through an API

Read this in small pieces. Explain each new term in your own words, trace the example on paper, and change one input before continuing. If the arithmetic is unfamiliar, use [the math bridge](../MATH_START_HERE.md).

### Concepts in plain language

A client sends an HTTP request: method, path, headers, and optional body. GET reads; POST submits data for processing/creation under the API contract. JSON represents structured values. A response contains a status code and often a JSON body.

The server first identifies the caller (authentication), checks allowed resources (authorization), validates fields, performs repository work, and serializes a response. A tenant is an organization or isolated data scope. Trusted tenant identity comes from authentication, not a field the caller can choose.

FastAPI declares routes and Pydantic validates request schemas. A dependency supplies a shared capability such as authentication. Persistence means data survives a new application process because it is stored outside that process's memory.

### Worked example: follow the values

A valid POST /records with a category and integer cents should create one tenant-scoped row and return 201. A negative amount should return 422 without changing the ledger. A missing credential should return 401. Restart the app and read the same row to establish persistence.

### Pause and explain

Why is a successful response from a dictionary stored in memory insufficient evidence of persistence?

<details><summary>Check your reasoning after trying</summary>

The dictionary disappears with the process. Reopen the database through a new app instance and verify the stored result.

</details>

### Common mistakes to check

An HTTP 200 alone does not prove the returned content is correct or authorized. Verify both status and body.

**Ready to continue?** Explain the example without reading it, predict what a changed input will do, and identify one invalid or missing input. Then attempt the original notebook exercises. You do not need to finish a whole notebook in one session.


## Lesson

HTTP requests have a method, path, headers, and optional body. GET reads; POST creates or triggers work; PUT usually replaces; PATCH partially updates; DELETE removes. Status codes communicate outcomes: 2xx success, 4xx client/request failures, 5xx server failures. A JSON response still needs a schema.

FastAPI maps functions to routes and Pydantic validates input. Dependency injection supplies shared resources such as database sessions and the authenticated principal. Middleware can record request duration or attach a request ID. Authentication establishes identity; authorization decides what that identity may access. Derive tenant scope on the server from verified identity, never from a caller-supplied tenant field.

Use tests at the HTTP boundary to verify status codes, bodies, malformed inputs, and permissions. An in-memory list is useful for this first exercise but loses data on restart and is not safe shared storage for multiple workers.


## Worked example

Predict the output before running. Change one input and explain the result.


In [ ]:
from fastapi import FastAPI
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field
app = FastAPI()
@app.get("/health")
def health():
    return {"status": "ok"}
assert TestClient(app).get("/health").json() == {"status": "ok"}


## Exercise 1: Request schema

Define Record(BaseModel) with category length 1..50 and amount finite >=0.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
from pydantic import BaseModel, Field
class Record(BaseModel):
    category: str = Field(min_length=1, max_length=50)
    amount: float = Field(ge=0, allow_inf_nan=False)


In [ ]:
assert Record(category="Food", amount=2).amount == 2
from pydantic import ValidationError
expect_error(ValidationError, lambda: Record(category="", amount=-1))
print("Exercise 1: checks passed")


**Why this works:** Validation expresses constraints centrally and generates an API schema.


## Exercise 2: Create endpoint

Write make_app() with POST /records returning the validated record with status 201.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def make_app():
    app = FastAPI()
    @app.post("/records", status_code=201)
    def create(record: Record):
        return record
    return app


In [ ]:
client = TestClient(make_app())
assert client.post("/records", json={"category": "Food", "amount": 2}).status_code == 201
assert client.post("/records", json={"category": "", "amount": -1}).status_code == 422
print("Exercise 2: checks passed")


**Why this works:** HTTP tests prove invalid input is rejected at the boundary, not just inside a helper.


## Exercise 3: Pagination contract

Return a slice for offset>=0 and 1<=limit<=100; reject other bounds.

Write down one normal case, one boundary case, and one invalid case before coding.


In [ ]:
def paginate(rows, offset=0, limit=20):
    if offset < 0 or not 1 <= limit <= 100: raise ValueError("Invalid pagination")
    return rows[offset:offset+limit]


In [ ]:
assert paginate(list(range(10)), 3, 2) == [3, 4]
expect_error(ValueError, lambda: paginate([], 0, 1000))
print("Exercise 3: checks passed")


**Why this works:** A production database uses stable ordering and LIMIT/OFFSET or cursor pagination rather than loading all rows.


## Independent transfer

Build Stage 03’s persistent API with GET /records, /summary, /statistics and POST /records. Add auth, request IDs, tests, and PostgreSQL ingestion. Restart it and prove persistence.

Record your implementation, errors, measurements, and explanation in work/. See the project’s ANSWERS.md after attempting the brief.


## Test your independent assignment

The assignment is work you build separately from the small notebook exercises. Use your own Python file under `work/`; notebooks remain useful for exploration and notes.

1. Read the [03c interface and acceptance cases](../assignments/03c.md).
2. Copy the [blank starter](../assignments/starters/03c.py) to `work/assignments/03c.py`, or adapt your existing implementation to the documented interface. Do not overwrite earlier work.
3. Implement one piece, then run this command **in your terminal from the repository root**, with your course environment activated:

```bash
python tools/test_assignment.py 03c --solution work/assignments/03c.py
```

The runner tests the file you specify. It never silently substitutes the reference answer. `NotImplementedError` is expected while the starter is incomplete. Read a failed assertion, calculate its expected answer by hand, fix your implementation, and rerun. See [how to read tests](../assignments/README.md) if pytest is new to you.

After a serious attempt, compare the [worked implementation](../answers/03c.md#03c-t). To check the provided answer explicitly:

```bash
python tools/test_assignment.py 03c --reference
```

**Completion:** pass the relevant automated cases, complete the additional experiment/live-evidence checklist in the assignment guide, and explain your approach without the answer open. An automated pass does not certify a live server, a deployment, a human review, or independent mastery.


## Explain without code

1. What does 422 mean here?

   The request body fails the declared validation contract.

2. Why inject a database session?

   It makes resource lifetime explicit and allows test substitution.


## Reading and review

- [Primary reference 1](https://fastapi.tiangolo.com/tutorial/)

After 2, 7, and 30 days: explain the concept from memory, solve a changed-input version, and log any gaps in progress.md.
